I used this website to get my authentication token for ngrok:
https://dashboard.ngrok.com/get-started/your-authtoken

In [10]:
# Cell 1 — Write all project files to disk
import os
os.makedirs("/content/abtf_app", exist_ok=True)

# ── filter.py ────────────────────────────────────────────────────────────────
filter_code = '''
import numpy as np
from scipy.ndimage import gaussian_filter
from skimage import color
import cv2, time

def compute_gradient_map(img_gray, sigma=1.0):
    blurred = gaussian_filter(img_gray, sigma=sigma)
    dx = np.gradient(blurred, axis=1)
    dy = np.gradient(blurred, axis=0)
    return np.sqrt(dx**2 + dy**2)

def compute_structure_map(G, m=4):
    H, W  = G.shape
    S     = np.zeros_like(G)
    sigma = max(m - 1, 1)
    directions = {
        "NW": (range(-m, 1), range(-m, 1)),
        "NE": (range(-m, 1), range(0, m+1)),
        "SW": (range(0, m+1), range(-m, 1)),
        "SE": (range(0, m+1), range(0, m+1)),
    }
    G_pad = np.pad(G, m, mode="reflect")
    for name, (row_rng, col_rng) in directions.items():
        rr, cc = np.meshgrid(list(row_rng), list(col_rng), indexing="ij")
        w      = np.exp(-(rr**2 + cc**2) / (2 * sigma**2))
        w     /= w.sum()
        ri = [r + m for r in row_rng]
        ci = [c + m for c in col_rng]
        for i in range(H):
            for j in range(W):
                patch = G_pad[i: i + 2*m + 1, j: j + 2*m + 1]
                sub   = patch[np.ix_(ri, ci)]
                A     = float(np.sum(w * sub))
                if A > S[i, j]:
                    S[i, j] = A
    s_min, s_max = S.min(), S.max()
    if s_max > s_min:
        S = (S - s_min) / (s_max - s_min)
    return S

def compute_scale_map(S, eta=10, lam=10, delta=1):
    W = eta * (1.0 / lam) ** (S**2)
    W = np.clip(np.round(W).astype(int), delta, eta)
    return W

def compute_fourier_coeffs(sigma_r, T, eps=0.01):
    v      = np.pi / T
    t_vals = np.linspace(-T, T, 2000)
    phi    = np.exp(-t_vals**2 / (2 * sigma_r**2))
    N = 1
    while N < 150:
        ns  = np.arange(-N, N+1)
        c_n = np.array([
            np.trapz(phi * np.exp(-1j * n * v * t_vals), t_vals) / (2*T)
            for n in ns
        ])
        phi_h = np.sum(
            [c_n[i] * np.exp(1j * ns[i] * v * t_vals) for i in range(len(ns))],
            axis=0
        ).real
        if np.max(np.abs(phi - phi_h)) <= eps:
            break
        N += 1
    return N, v, c_n

def _build_integral_image(r):
    R = np.zeros((r.shape[0]+1, r.shape[1]+1), dtype=r.dtype)
    R[1:, 1:] = np.cumsum(np.cumsum(r, axis=0), axis=1)
    return R

def _box_sum(R, i, j, wp):
    H, W = R.shape[0]-1, R.shape[1]-1
    r1, r2 = max(i-wp, 0), min(i+wp, H-1)
    c1, c2 = max(j-wp, 0), min(j+wp, W-1)
    return R[r2+1, c2+1] - R[r1, c2+1] - R[r2+1, c1] + R[r1, c1]

def _filter_channel(f, W_map, N, v, c_n):
    H, W = f.shape
    ns   = np.arange(-N, N+1)
    E    = np.zeros((H, W), dtype=complex)
    Hd   = np.zeros((H, W), dtype=complex)
    for idx, n in enumerate(ns):
        cn        = c_n[idx]
        exp_q     = np.exp(1j * n * v * f)
        R_h       = _build_integral_image(exp_q)
        R_e       = _build_integral_image(f * exp_q)
        exp_neg_p = np.exp(-1j * n * v * f)
        en = np.zeros((H, W), dtype=complex)
        hn = np.zeros((H, W), dtype=complex)
        for wp in np.unique(W_map):
            rows, cols = np.where(W_map == wp)
            for k in range(len(rows)):
                i, j     = rows[k], cols[k]
                en[i, j] = _box_sum(R_e, i, j, wp)
                hn[i, j] = _box_sum(R_h, i, j, wp)
        E  += cn * exp_neg_p * en
        Hd += cn * exp_neg_p * hn
    out = np.real(E) / (np.real(Hd) + 1e-8)
    return np.clip(out, 0, 255)

def _filter_rgb(img, W_map, N, v, c_n):
    out = np.zeros_like(img)
    for ch in range(3):
        out[:, :, ch] = _filter_channel(img[:, :, ch], W_map, N, v, c_n)
    return out

def run_filter(pil_image, sigma_r=35.0, eta=10, lam=10, delta=1,
               iterations=2, max_dim=400, progress_callback=None):
    def _cb(name, pct):
        if progress_callback:
            progress_callback(name, pct)

    timings = {}
    img_np  = np.array(pil_image.convert("RGB"), dtype=np.float64)
    h, w    = img_np.shape[:2]
    if max(h, w) > max_dim:
        scale  = max_dim / max(h, w)
        img_np = cv2.resize(img_np, (int(w*scale), int(h*scale)),
                            interpolation=cv2.INTER_AREA)
    img_gray = color.rgb2gray(img_np / 255.0) * 255.0
    _cb("Resized image", 0.05)

    t0 = time.time()
    G  = compute_gradient_map(img_gray, sigma=1.0)
    timings["gradient_map"] = time.time() - t0
    _cb("Gradient map", 0.15)

    t0 = time.time()
    S  = compute_structure_map(G, m=4)
    timings["structure_map"] = time.time() - t0
    _cb("Structure map", 0.50)

    t0 = time.time()
    W_map = compute_scale_map(S, eta=eta, lam=lam, delta=delta)
    timings["scale_map"] = time.time() - t0
    _cb("Scale map", 0.55)

    u = img_np.copy()
    timings["filtering"] = 0.0
    for it in range(iterations):
        T_val = max(float(np.max(u) - np.min(u)), 1e-6)
        N_f, v_f, c_f = compute_fourier_coeffs(sigma_r, T_val, eps=0.01)
        t0 = time.time()
        u  = _filter_rgb(u, W_map, N_f, v_f, c_f)
        timings["filtering"] += time.time() - t0
        _cb(f"Filtering iteration {it+1}/{iterations}", 0.55 + 0.40*(it+1)/iterations)

    timings["total"] = sum(timings.values())
    _cb("Done", 1.0)

    return {
        "input":     img_np.astype(np.uint8),
        "gradient":  G,
        "structure": S,
        "scale":     W_map,
        "result":    np.clip(u, 0, 255).astype(np.uint8),
        "timings":   timings,
    }
'''

# ── app.py ───────────────────────────────────────────────────────────────────
app_code = '''
import io, sys
sys.path.insert(0, "/content/abtf_app")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import streamlit as st
from filter import run_filter

st.set_page_config(page_title="Adaptive Bilateral Texture Filter",
                   page_icon="🖼️", layout="wide")

def _to_png_bytes(arr):
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return buf.getvalue()

def _map_figure(data, title, cmap, colorbar_label=None):
    fig, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axis("off")
    if colorbar_label:
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=colorbar_label)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf

# ── Title ────────────────────────────────────────────────────────────────────
st.title("  Adaptive Bilateral Texture Filter")
st.markdown("""
**Paper:** *Adaptive Bilateral Texture Filter for Image Smoothing* — Xu et al., 2022
Upload an image. The filter **removes textures** while **keeping structural edges** sharp.
""")
st.divider()

# ── Sidebar ──────────────────────────────────────────────────────────────────
with st.sidebar:
    st.header(" Parameters")
    sigma_r    = st.slider("σ_r — Range kernel strength", 10, 60, 35, 5,
                           help="Higher = smoother result. Paper suggests 20–40.")
    eta        = st.slider("η — Max window size (px)", 4, 20, 10, 2,
                           help="Larger = more smoothing in texture areas. Paper suggests 8–16.")
    iterations = st.slider("Iterations", 1, 3, 2,
                           help="More passes = smoother but slower. Paper default is 2.")
    max_dim    = st.select_slider("Max image size (px)", [200, 300, 400, 500], 400,
                                  help="Longer edge is resized to this. Smaller = faster.")
    st.divider()
    st.caption("Fixed: λ=10, δ=1, ε=0.01")

# ── Upload ───────────────────────────────────────────────────────────────────
uploaded = st.file_uploader("📂  Upload an image (JPG or PNG)", type=["jpg","jpeg","png"])
if uploaded is None:
    st.info("  Upload an image above to get started.")
    st.stop()

pil_image = Image.open(uploaded)

# ── Run filter ───────────────────────────────────────────────────────────────
st.subheader("Processing…")
bar    = st.progress(0)
status = st.empty()

@st.cache_data(show_spinner=False)
def _run(img_bytes, sigma_r, eta, iterations, max_dim):
    return run_filter(Image.open(io.BytesIO(img_bytes)),
                      sigma_r=sigma_r, eta=eta, lam=10, delta=1,
                      iterations=iterations, max_dim=max_dim)

out = _run(uploaded.getvalue(), sigma_r, eta, iterations, max_dim)
bar.progress(1.0)
status.text("  Done!")
timings = out["timings"]

# ── Before / After ───────────────────────────────────────────────────────────
st.divider()
st.subheader("  Before & After")
c1, c2 = st.columns(2)
c1.image(out["input"],  caption="Input Image",             use_container_width=True)
c2.image(out["result"], caption="Filtered — Textures Removed", use_container_width=True)
st.download_button("⬇  Download filtered image",
                   _to_png_bytes(out["result"]), "filtered.png", "image/png")

# ── Intermediate maps ─────────────────────────────────────────────────────────
st.divider()
st.subheader(" Pipeline — Intermediate Maps")
st.caption("These four maps show what happens inside the algorithm (Figure 1 of the paper).")
m1, m2, m3, m4 = st.columns(4)
m1.image(out["input"],
         caption="(A) Input", use_container_width=True)
m2.image(_map_figure(out["gradient"],  "(B) Gradient Map",  "gray"),
         caption="Bright = strong gradient", use_container_width=True)
m3.image(_map_figure(out["structure"], "(C) Structure Map", "hot"),
         caption="Bright = edge / Dark = texture", use_container_width=True)
m4.image(_map_figure(out["scale"],     "(D) Scale Map",     "viridis",
                     colorbar_label="Window radius (px)"),
         caption="Bright = large window / Dark = small window", use_container_width=True)

# # ── Detail enhancement ───────────────────────────────────────────────────────
# st.divider()
# st.subheader("✨  Bonus: Detail Enhancement")
# st.caption("Input + 3× (Input − Filtered) → sharpens fine details instead of removing them.")
# residual = out["input"].astype(np.float64) - out["result"].astype(np.float64)
# enhanced = np.clip(out["input"].astype(np.float64) + 3.0*residual, 0, 255).astype(np.uint8)
# d1, d2 = st.columns(2)
# d1.image(out["input"], caption="Input", use_container_width=True)
# d2.image(enhanced, caption="Detail Enhanced", use_container_width=True)
# st.download_button("⬇  Download detail-enhanced image",
#                    _to_png_bytes(enhanced), "detail_enhanced.png", "image/png")

# ── Timing ───────────────────────────────────────────────────────────────────
st.divider()
st.subheader("⏱️  Processing Time")
import pandas as pd
labels = {
    "gradient_map":  "Gradient Map  (Eq. 3)",
    "structure_map": "Structure Map  (Eq. 4)",
    "scale_map":     "Scale Map  (Eq. 5)",
    "filtering":     f"Texture Filtering ×{iterations}  (Eq. 9–21)",
    "total":         "Total",
}
df = pd.DataFrame([{"Step": labels[k], "Time": f"{timings.get(k,0):.2f}s"}
                   for k in labels])
t1, t2 = st.columns(2)
t1.dataframe(df, hide_index=True, use_container_width=True)

keys   = ["gradient_map","structure_map","scale_map","filtering"]
values = [timings.get(k,0) for k in keys]
fig_p, ax_p = plt.subplots(figsize=(4,4))
ax_p.pie(values, labels=["Gradient","Structure","Scale","Filtering"],
         autopct="%1.0f%%", startangle=140,
         colors=["#4c72b0","#c44e52","#55a868","#dd8452"])
ax_p.set_title("Time share", fontsize=11)
buf_p = io.BytesIO()
fig_p.savefig(buf_p, format="png", dpi=120, bbox_inches="tight")
plt.close(fig_p); buf_p.seek(0)
t2.image(buf_p, use_container_width=True)

st.divider()
st.caption("Xu H, Zhang Z, Gao Y, Liu H, Xie F and Li J (2022) "
           "Adaptive Bilateral Texture Filter for Image Smoothing. "
           "Front. Neurorobot. 16:729924.")
'''

# ── Write both files ──────────────────────────────────────────────────────────
with open("/content/abtf_app/filter.py", "w") as f:
    f.write(filter_code.strip())

with open("/content/abtf_app/app.py", "w") as f:
    f.write(app_code.strip())

print(" Files written:")
print("   /content/abtf_app/filter.py")
print("   /content/abtf_app/app.py")

 Files written:
   /content/abtf_app/filter.py
   /content/abtf_app/app.py


In [11]:
# Cell 2 — Install packages
# streamlit-related packages not pre-installed in Colab
!pip install streamlit pyngrok --quiet
print(" Installed streamlit and pyngrok")

 Installed streamlit and pyngrok


In [12]:
# Cell 3 — Connect ngrok (paste your token here)
from pyngrok import ngrok

# ← Paste your ngrok token between the quotes below
NGROK_TOKEN = "2wgqD1VOzKcaP3Rs0mxzQQk9WtQ_2nFYBAcJAhafxF4xscFmA"

ngrok.set_auth_token(NGROK_TOKEN)
print(" ngrok authenticated")

 ngrok authenticated


In [13]:
# Cell 4 — Start Streamlit and get the public URL
import subprocess, threading, time
from pyngrok import ngrok

# Kill any leftover streamlit processes from previous runs
!pkill -f streamlit 2>/dev/null || true
time.sleep(1)

# Start Streamlit in the background on port 8501
proc = subprocess.Popen(
    ["streamlit", "run", "/content/abtf_app/app.py",
     "--server.port", "8501",
     "--server.headless", "true",          # no browser pop-up attempt
     "--server.enableCORS", "false",       # required for ngrok tunnel
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give Streamlit a few seconds to start
time.sleep(4)

# Open a public tunnel to port 8501
tunnel = ngrok.connect(8501)
print("=" * 55)
print("  Your app is live at:")
print(f"  👉  {tunnel.public_url}")
print("=" * 55)
print("  Open that link in any browser.")
print("  Keep this Colab tab open while using the app.")
print("  Re-run this cell if the link expires (ngrok free = 2h limit).")

^C


  Your app is live at:
  👉  https://31f7-104-199-178-61.ngrok-free.app
  Open that link in any browser.
  Keep this Colab tab open while using the app.
  Re-run this cell if the link expires (ngrok free = 2h limit).
